# Avida CFO – contract extraction benchmark (OpenShift AI Workbench)

The app lets the owner **upload a contract**; Granite on vLLM extracts the commercial terms live and the
**Legal Agent** checks every price increase / late payment against them.

This notebook is the **evaluation harness for exactly that code**: it imports `backend/app/extraction.py`
(the same prompt, JSON schema, guided decoding and hallucination guards the backend runs) and measures it
against the hand-labelled `backend/data/contracts_index.json`.

1. Load the contracts (`.md` split into pages, `.pdf` via `pypdf` – same loader as the upload endpoint)
2. Extract with the production extractor on the vLLM model in this cluster
3. Benchmark field by field (the fields the Legal Agent's rules read) + clause recall + latency
4. Optionally regenerate the index, or send a contract to the running app

Run it in a workbench in the same cluster (the internal service URLs work there).

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import json, os, sys, time
from pathlib import Path
from openai import OpenAI

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = REPO / "backend" / "data"
sys.path.insert(0, str(REPO / "backend"))
from app import extraction  # <- the code the backend runs for every upload

LLM_BASE_URL = os.getenv("LLM_BASE_URL", "http://granite-8b-predictor.avida-models.svc.cluster.local:8080/v1")
LLM_MODEL = os.getenv("LLM_MODEL", "granite-8b")
client = OpenAI(base_url=LLM_BASE_URL, api_key=os.getenv("LLM_API_KEY", "EMPTY"))
OWN_COMPANY = json.loads((DATA / "company.json").read_text(encoding="utf-8"))["name"]
print("models:", [m.id for m in client.models.list().data], "| our company:", OWN_COMPANY)

## 1. Load the contracts (same loader as `POST /api/contracts/upload`)

In [ ]:
docs = {}
for f in sorted((DATA / "contracts").rglob("*")):
    if f.suffix.lower() in (".md", ".txt", ".pdf"):
        docs[f.relative_to(DATA).as_posix()] = extraction.pages_from_upload(f.name, f.read_bytes())
{k: len(v) for k, v in docs.items()}

## 2. Extract with the production extractor

Prompt and schema below are imported from the backend – change them there, re-run this notebook, compare.

In [ ]:
print(extraction.system_prompt(OWN_COMPANY))
print()
print(json.dumps(extraction.SCHEMA["properties"], indent=1)[:600], "…")

In [ ]:
extracted, latency = {}, {}
for name, pages in docs.items():
    t0 = time.perf_counter()
    extracted[name] = extraction.extract_sync(client, LLM_MODEL, pages, OWN_COMPANY)
    latency[name] = round(time.perf_counter() - t0, 1)
    ex = extracted[name]
    print(f"{name:45} {latency[name]:5}s  {ex['role']:8} {ex['counterparty']:25} clauses={len(ex['clauses'])} warnings={ex['warnings']}")

## 3. Benchmark against the hand-labelled index

In [ ]:
truth = json.loads((DATA / "contracts_index.json").read_text(encoding="utf-8"))["contracts"]
rows = []
for name, ex in extracted.items():
    gt = extraction.match_truth(ex["counterparty"], truth)
    if not gt:
        print("no ground truth for", name, ex["counterparty"]); continue
    rows += extraction.compare(ex, gt)

w = max(len(r["party"]) for r in rows)
for r in rows:
    print(f"{r['party']:{w}} | {r['field']:34} | expected {str(r['expected']):12} | got {str(r['got']):12} | {'OK' if r['ok'] else 'MISMATCH'}")

hits = sum(r["ok"] for r in rows)
summary = {
    "model": LLM_MODEL,
    "contracts": len(extracted),
    "field_accuracy": round(hits / len(rows), 3),
    "fields": f"{hits}/{len(rows)}",
    "avg_latency_s": round(sum(latency.values()) / len(latency), 1),
    "guard_warnings": sum(len(e["warnings"]) for e in extracted.values()),
}
(REPO / "notebooks" / "benchmark_results.json").write_text(json.dumps({**summary, "rows": rows}, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
summary

## 4. (Optional) regenerate the index used by the backend

Keeps ids/aliases from the existing file. Set `OVERWRITE = True` only if the benchmark looks good.

In [ ]:
OVERWRITE = False

out = []
for i, (name, ex) in enumerate(sorted(extracted.items()), start=1):
    gt = extraction.match_truth(ex["counterparty"], truth) or {}
    out.append(extraction.to_index_entry(ex, gt.get("id", f"C-{i:03d}"), name, gt.get("aliases")))

target = DATA / ("contracts_index.json" if OVERWRITE else "contracts_index.generated.json")
target.write_text(json.dumps({"_comment": "Generated by notebooks/contract_extraction.ipynb", "contracts": out},
                             indent=2, ensure_ascii=False), encoding="utf-8")
print("written", target)

## 5. (Optional) upload a contract to the running app

The same endpoint the UI uses. After this, press **Check my inbox** in the app: the Legal Agent's trace says
*"contract you uploaded, terms extracted by the model"*.

In [ ]:
import httpx

APP_URL = os.getenv("APP_URL", "http://avida-cfo.avida-cfo.svc.cluster.local:8080")
SEND = "contracts/stahlhandel_framework.md"   # any file from `docs`

r = httpx.post(f"{APP_URL}/api/contracts/upload", files={"file": (Path(SEND).name, (DATA / SEND).read_bytes())}, timeout=120)
print(r.status_code)
r.json()